# The ML workflow and the train/test split

**Video 9.2** · ShopKart project

**Goal:** Turn the ShopKart table into X (features) and y (target), and split it into a training set and a test set – the first step of every machine-learning project.

**How to open in Colab:** go to colab.research.google.com → File → Upload notebook → choose this file. Then click the folder icon on the left and drag these data files in: `shopkart_clean.csv`. Then run cells from top to bottom (Shift + Enter).

## The workflow we follow in every ML video
1. Ask a question (what do we want to predict?)
2. Choose the target **y** and the features **X**
3. Split into train and test
4. Train (fit) a model on train
5. Predict on test
6. Evaluate – is it good enough to use?

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("shopkart_clean.csv")
df.shape

(5000, 21)

## Step 1 – the question
**Will this order be returned?** (We build the actual model in Video 9.4.) Each row is one order, so each row is one example the model can learn from.

## Step 2 – target (y) and features (X)
- **y** = the column we want to predict → `returned`
- **X** = the columns we use as clues → category, city, payment mode, delivery days, discount, membership, age, price

In [2]:
# y: turn "Y"/"N" into 1/0 because models need numbers
y = (df["returned"] == "Y").astype(int)
print(y.value_counts())
print("Share returned:", round(y.mean() * 100, 1), "%")

returned
0    4313
1     687
Name: count, dtype: int64
Share returned: 13.7 %


In [3]:
features = ["category", "city", "payment_mode", "delivery_days",
            "discount", "membership", "age", "unit_price"]
X_raw = df[features]
X_raw.head()

,category,city,payment_mode,delivery_days,discount,membership,age,unit_price
0,Books,Delhi,COD,4,0.20,Prime,26,362
1,Beauty,Chennai,Debit Card,4,0.05,Free,35,300
2,Fashion,Delhi,Debit Card,2,0.05,Prime,41,2258
3,Home,Lucknow,COD,4,0.15,Free,35,2020
4,Home,Chennai,UPI,2,0.10,Free,33,1937


## Models only understand numbers
`category` has text like "Fashion". `pd.get_dummies` turns one text column into several 0/1 columns (this is called **one-hot encoding**). `drop_first=True` drops one column per group because it is redundant (if all the others are 0, it must be the dropped one).

In [4]:
# Small demo on 5 rows so you can see what happens
pd.get_dummies(df[["category"]].head(5), dtype=int)

,category_Beauty,category_Books,category_Fashion,category_Home
0,0,1,0,0
1,1,0,0,0
2,0,0,1,0
3,0,0,0,1
4,0,0,0,1


In [5]:
X = pd.get_dummies(X_raw, drop_first=True, dtype=int)
print("Columns before:", X_raw.shape[1], " after:", X.shape[1])
list(X.columns)

Columns before: 8  after: 23


['delivery_days',
 'discount',
 'age',
 'unit_price',
 'category_Books',
 'category_Electronics',
 'category_Fashion',
 'category_Home',
 'city_Bengaluru',
 'city_Chennai',
 'city_Delhi',
 'city_Hyderabad',
 'city_Jaipur',
 'city_Kolkata',
 'city_Lucknow',
 'city_Mumbai',
 'city_Pune',
 'payment_mode_Credit Card',
 'payment_mode_Debit Card',
 'payment_mode_Net Banking',
 'payment_mode_UPI',
 'payment_mode_Unknown',
 'membership_Prime']

## Step 3 – split into train and test
Think of an exam. If you practise with the exact questions that come in the exam, your score means nothing. So we **hide 20% of orders** (test set) and train only on the other 80%.

- `test_size=0.2` → 20% test
- `random_state=42` → the same random split every time (so your numbers match mine)
- `stratify=y` → keep the same share of returns (~14%) in train and test

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train rows:", len(X_train), " Test rows:", len(X_test))
print("Return share in train:", round(y_train.mean() * 100, 2), "%")
print("Return share in test :", round(y_test.mean() * 100, 2), "%")

Train rows: 4000  Test rows: 1000
Return share in train: 13.75 %
Return share in test : 13.7 %


## Recap
- y = what we predict, X = clues
- Text → numbers with `get_dummies`
- Train on 80%, test on the hidden 20%
- `random_state=42` so results are repeatable

## Bonus (not shown in the video): why random_state matters
The video only mentions this in one sentence. Change the number and the rows in the test set change. Run this cell to see it: with `random_state=42` your split (and every score) matches the video.

In [7]:
for seed in [1, 42, 7]:
    a, b, c, d = train_test_split(X, y, test_size=0.2, random_state=seed)
    print("random_state =", seed, "-> first 5 test row numbers:", list(b.index[:5]))

random_state = 1 -> first 5 test row numbers: [2764, 4767, 3814, 3499, 2735]
random_state = 42 -> first 5 test row numbers: [1501, 2586, 2653, 1055, 705]
random_state = 7 -> first 5 test row numbers: [3406, 757, 3624, 4544, 3235]
